<a href="https://colab.research.google.com/github/BillPapakyriakou/DataMining-Notebooks/blob/main/assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Τρίτη Σειρά Ασκήσεων
---
### 5324 - Παπακυριακού Βασίλειος


---
##**Ερώτηση 1**


##1.
### **Απόδειξη**:

$$
p_v^T = (1-a)p_v^T P + a v^T
$$

Aπό διαφάνεια 47 (datamining-LAR.pdf),
λύνοντας ως προς $p_v$ έχουμε:

$$
p_v^T = a v^T \left(I - (1-a)P\right)^{-1}
$$

Αντικαθιστούμε στη σχέση  $p_v^T = v^T Q$:

$$
a v^T \left(I - (1-a)P\right)^{-1} = v^T Q
$$

Λύνουμε ως πρός Q:

$$
\boxed{
Q = a \left(I - (1-a)P\right)^{-1}
}
$$

##2.
### **Απόδειξη**:

Έχουμε τη σχέση:  

$$
p_v^T = v^T Q
$$

Για το personalized PageRank έχουμε jump vector:

$$
v^T = (0,\ldots,0,1,0,\ldots,0).
$$

Άρα:

$$
p_i^T = (0,\ldots,0,1,0,\ldots,0)\, Q,
$$

που είναι η i-οστή γραμμή του $Q$.

$$
\fbox{Άρα η i-οστή γραμμή του $Q$ είναι το personalized PageRank διάνυσμα $p_i^T$.}
$$




##3.
### **Απόδειξη**:

Έχουμε το ομοιόμορφο jump vector:

$$
u^T = \left(\frac{1}{n}, \frac{1}{n}, \ldots, \frac{1}{n}\right).
$$

Από τη σχέση $ p_u^T = u^T Q $ :

$$
p_u^T = \left(\frac{1}{n}, \frac{1}{n}, \ldots, \frac{1}{n}\right) Q.
$$

Αυτό αναπαριστά τον μέσο όρο των γραμμών του $Q$.

Από το ερώτημα 2, δείξαμε ότι οι γραμμές του $Q$ είναι τα personalized PageRank διανύσματα $p_i^T$, άρα έχουμε:

$$
p_u^T = \frac{1}{n} \sum_{i=1}^n p_i^T.
$$

Σε μορφή στηλών, γράφεται ως:

$$
\boxed{
p_u = \frac{1}{n} \sum_{i=1}^n p_i.
}
$$


##4.
### **Απόδειξη**:

Θεωρούμε ένα οποιοδήποτε jump vector

$$
v = (v(1), v(2), v(3), \ldots, v(n)),
$$

με $v(i) \ge 0$ και $\sum_{i=1}^n v(i) = 1.$

Από τη σχέση $p_v^T = v^T Q$ έχουμε:

$$
p_v^T = (v(1), v(2), v(3), \ldots, v(n))\, Q
$$

Που είναι γραμμικός συνδυασμός των γραμμών του $Q$ με συντελεστές $v(i)$:

$$
p_v^T = \sum_{i=1}^n v(i)\, (\text{$i$-οστή γραμμή του } Q).
$$

Από το ερώτημα 2, η $i$-οστή γραμμή του $Q$ είναι το personalized PageRank διάνυσμα $p_i^T$, άρα.

$$
p_v^T = \sum_{i=1}^n v(i)\, p_i^T.
$$

Σε μορφή στηλών γράφεται ως:
$$
\boxed{
p_v = \sum_{i=1}^n v(i)\, p_i.
}
$$

---
##**Ερώτηση 2**


In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

# φόρτωση δεδομένων
ratings = pd.read_csv("ratings_small.csv", low_memory=False)
links = pd.read_csv("links_small.csv", low_memory=False)
movies = pd.read_csv("movies_metadata.csv", low_memory=False)

# data cleaning - βγάζουμε NaN, κρατάμε μόνο valid Ids (int)

links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce")
movies["id"] = pd.to_numeric(movies["id"], errors="coerce")

links = links.dropna(subset=["tmdbId"])
movies = movies.dropna(subset=["id"])

links["tmdbId"] = links["tmdbId"].astype(int)
movies["id"] = movies["id"].astype(int)


# κάνουμε join τα ratings με τα links και τα movies
dataframe = ratings.merge(links, on="movieId", how="inner")
dataframe = dataframe.merge(
    movies[["id", "title"]],
    left_on="tmdbId",
    right_on="id",
    how="inner"
)

# κρατάμε userId, movieId και το rating του user για την ταινία
dataframe = dataframe[["userId", "id", "rating"]]


counts = dataframe.groupby("id").size()  # μετράμε τον αριθμό ratings ανά ταινία
valid_movies = counts[counts >= 10].index  # κρατάμε τα ids που έχουν τουλάχιστον 10 ratings
dataframe = dataframe[dataframe["id"].isin(valid_movies)]  # αφήνουμε μόνο τις ταινίες με >=10 ratings στα δεδομένα

# factorize για να φτιαχτεί σωστά ο sparse πίνακας - δουλεύει με indices 0...n και όχι με ids, αν δεν το κάνουμε
#                                                                             ο πίνακας θα βγεί πολύ μεγαλύτερος
# πχ ids = pd.Series([349, 999, 349, 1010])
# codes, uniques = pd.factorize
# codes = [0, 1, 0, 2], uniques = [349, 999, 1010]

# sort για να παίρνουμε κάθε φορά τα ίδια mappings
movie_codes, movie_ids = pd.factorize(dataframe["id"], sort=True)
user_codes, user_ids = pd.factorize(dataframe["userId"], sort=True)

# δημιουργούμε αραιό πίνακα (ταινίες-χρήστες-ratings)
R = csr_matrix(
    (dataframe["rating"].to_numpy(np.float32), (movie_codes, user_codes)),
    shape=(len(movie_ids), len(user_ids))
)

print("R shape:", R.shape)


R shape: (2242, 671)
